黑色星期五数据集来自于Kaggle网，本次基于该数据集对黑五情况其进行分析

数据简介：

-user_id: 用户id
-product_id: 商品id
-gender: 性别，F女M男
-age: 7个年龄段
-occupation: 用0-20表示
-city_category: 城市类别 A、B、C
-stay_in_current_city_years: 居住城市时长 0，1,2,3,4+
-martial_status: 婚姻状况 0：已婚；1：未婚
-product_category_1：主要产品类别（不存在空值）
-product_category_2：次要产品类别（有空值）
-product_category_3：次要产品类别（有空值）
-purchase: 购买的总金额

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

test=pd.read_csv('data/test.csv')
train=pd.read_csv('data/train.csv')
data = pd.concat([train, test], axis=0, ignore_index=True)

data.head()

In [ ]:
data.info()

In [ ]:
data.isnull().sum()

In [ ]:
(data.shape[0]- data.dropna(how='any').shape[0])/data.shape[0]

发现：

整体数据情况较为完整，仅次要类别2和3存在缺失,辨认为次要类别，可不进行处理
一、数据预处理

去除重复用户记录后，计算每个用户的总购买量，并将总购买量信息添加到对应的用户记录数据框

In [ ]:
df = data.drop_duplicates(subset=['User_ID'])[['User_ID', 'Gender','Age','Occupation','City_Category','Stay_In_Current_City_Years','Marital_Status']].sort_values(by='User_ID')
df['Purchase'] = data.groupby('User_ID')['Purchase'].sum().sort_index().values

In [ ]:
df.info()
df.head()

In [ ]:
df['Gender'] = df['Gender'].map({'F': 0, 'M': 1})
df['Age'] = df['Age'].map({'0-17':1,'18-25': 2,'26-35': 3,'36-45': 4,'46-50': 5,'51-55': 6,'55+':7})
df['City_Category'] = df['City_Category'].map({'A': 1,'B': 2,'C': 3})
df['Stay_In_Current_City_Years'] = df['Stay_In_Current_City_Years'].map({'0':0,'1':1,'2':2,'3':3,'4+':4})

df.head()

自身情况分析

In [ ]:
graph_by_variables = df.columns

plt.figure(figsize = (10, 10))
for i in range(1, 7):
    plt.subplot(2, 4, i+1)
    sns.histplot(data=df, x=graph_by_variables[i])
    plt.title(graph_by_variables[i])
plt.tight_layout()

整体可以看到：

1、男性购买数据多于女性；
 2、集中在36-45年龄段之间的用户最多，26-35和46-50之间的人数差不多；
 3、职业类型1、4、6的人进行购物较多，其中1最多；
 4、在城市C的购买量较大；
 5、大部分用户在当前城市生活时间在1年多；
 6、未婚和已婚的整体购买区别不明显，未婚人士购买量稍多于已婚。

In [ ]:
f, ax = plt.subplots(figsize = (7, 7))
sns.heatmap(df.corr(), annot = True, linewidths = 0.5, fmt = '.1f', ax = ax)

整体上相关性并不强

二、分类分析：

对于消费频次和消费金额双高的用户，认为是高价值用户，对于此数据引入测试数据集和训练数据集，以此来鉴别高价值客户

In [ ]:
# 高价值用户分析
purchase_counts = data['User_ID'].value_counts().reset_index()
purchase_counts.columns = ['User_ID', 'purchase_count']
total_amounts = data.groupby('User_ID')['Purchase'].sum().reset_index()
total_amounts.columns = ['User_ID', 'total_amount']
stats_df = purchase_counts.merge(total_amounts, on='User_ID')
stats_df_sorted_by_count = stats_df.sort_values(by='purchase_count', ascending=False)
stats_df_sorted_by_amount = stats_df.sort_values(by='total_amount', ascending=False)
df_count_top = stats_df_sorted_by_count.head(5000)
df_amount_top = stats_df_sorted_by_amount.head(5000)

common_users = pd.merge(df_count_top, df_amount_top, on='User_ID', how='inner')
merged_df = pd.merge(data, common_users, on='User_ID', how='inner')
gender_counts = merged_df.groupby('User_ID')['Gender'].value_counts()
 
# 高价值用户的男女比例
gender_cnt = gender_counts.groupby('Gender').size()
gender_cnt 

In [ ]:
high_value_user_ids = pd.merge(df_count_top[['User_ID']], df_amount_top[['User_ID']], on='User_ID', how='inner')
df = df.merge(high_value_user_ids.assign(is_value=1), on='User_ID', how='left')
df['is_value'] = df['is_value'].fillna(0)
df.head()

In [ ]:
df.isnull().sum()

In [ ]:
from sklearn.model_selection import train_test_split
# Splitting into train and test sets
X = df.drop(['User_ID','is_value'], axis = 1)
y = df['is_value']
# test_size = 0.2意为训练集占80%，测试集占20%，即将数据集按4:1的比例划分为训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 628)
print(len(X_train), len(X_test))

逻辑回归

In [ ]:
from sklearn.linear_model import LogisticRegression
# Fitting a logistic regression model with default parameters
logreg = LogisticRegression()
logreg.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# Prediction & Evaluation
y_hat_test = logreg.predict(X_test)
# Logistic Regression score
print("Logistic regression score for test set:")
print("Precision: {:.3f}".format(precision_score(y_test, y_hat_test)), "Recall: {:.3f}".format(recall_score(y_test, y_hat_test)))
print("F1 score: {:.3f}".format(f1_score(y_test, y_hat_test)))
print("AUC score: {:.3f}".format(roc_auc_score(y_test, y_hat_test)))

整体数据拟合良好

-约有 95.5% 的样本确实是正类，说明模型在预测为正类时的准确性比较高，误判为正类（即把负类预测成正类）的情况相对较少。
-实际的正类样本中，模型能够正确找出来的比例约为 97.3%，说明模型对正类样本的查全能力较强，遗漏正类样本（也就是把正类误判为负类）的情况较少。
-模型在精确率和召回率这两方面的综合表现较好，整体分类性能处于一个相对较高的水平。
-模型在区分不同类别方面有较好的表现，能够比较有效地将正类和负类样本区分开来，但距离完美的分类（AUC = 1）还有一定的提升空间。
决策树算法

In [ ]:
from sklearn.tree import DecisionTreeClassifier
# Fitting a decision tree model with default parameters
dt = DecisionTreeClassifier()
dt.fit(X_train, y_train)

In [ ]:
# Prediction & Evaluation
print("Decision tree score for test set:")
# Precision, Recall, F1-score
y_hat_test = dt.predict(X_test)
print("Precision: {:.3f}".format(precision_score(y_test, y_hat_test)), "Recall: {:.3f}".format(recall_score(y_test, y_hat_test)))
print("F1 score: {:.3f}".format(f1_score(y_test, y_hat_test)))
# AUC
y_hat_test = dt.predict_proba(X_test)
print("AUC score: {:.3f}".format(roc_auc_score(y_test, y_hat_test[:,1])))

In [ ]:
from sklearn import tree
import graphviz

dot_data = tree.export_graphviz(dt, feature_names=X_train.columns, filled=True, class_names=True,
                                proportion=True)
graph = graphviz.Source(dot_data)
graph

In [ ]:
from sklearn.tree import DecisionTreeClassifier
# Fitting a decision tree model with default parameters
dt = DecisionTreeClassifier(max_depth = 7)
dt.fit(X_train, y_train)

In [ ]:
# Prediction & Evaluation
y_hat_test = dt.predict(X_test)
# Decision Tree score
print("Decision tree score for test set:")
print("Predicion: {:.3f}".format(precision_score(y_test, y_hat_test)), "Recall: {:.3f}".format(recall_score(y_test, y_hat_test)))
print("F1 score: {:.3f}".format(f1_score(y_test, y_hat_test)))
print("AUC score: {:.3f}".format(roc_auc_score(y_test, y_hat_test)))

In [ ]:
from sklearn import tree
import graphviz
import os
dot_data = tree.export_graphviz(dt, feature_names = X.columns, filled = True, class_names = True,out_file=None)  
graph = graphviz.Source(dot_data)
if not os.path.exists('output'):
    os.makedirs('output')
graph.render('output/decision_tree.png')
graph

In [ ]:
import matplotlib.pyplot as plt 

max_depths = range(1, 15)
training_error = []
for max_depth in max_depths:
    model_1 = DecisionTreeClassifier(max_depth=max_depth)
    model_1.fit(X_train, y_train)
    training_error.append(roc_auc_score(y_train, model_1.predict(X_train)))
    
testing_error = []
for max_depth in max_depths:
    model_2 = DecisionTreeClassifier(max_depth=max_depth)
    model_2.fit(X_train, y_train)
    testing_error.append(roc_auc_score(y_test, model_2.predict(X_test)))

In [ ]:
plt.plot(max_depths, training_error, color='blue', label='Training AUC score')
plt.plot(max_depths, testing_error, color='green', label='Testing AUC score')
plt.xlabel('Tree depth')
plt.axvline(x=7, color='orange', linestyle='--')
plt.ylabel('roc_auc_score')
plt.title('Hyperparameter Tuning', pad=15, size=15)
plt.legend()
plt.savefig('error.png')

三、关联规则分析

In [ ]:
data['Product_Category_1'].nunique()

共有20种类别的商品，不妨对商品进行关联规则分析

In [ ]:
# 按照用户ID进行分组，然后对每个用户组内的商品ID进行聚合，收集每个用户购买的所有商品ID
grouped_data = data.groupby('User_ID')['Product_ID'].apply(list).reset_index()

# 重命名列名，使其更清晰
grouped_data.columns = ['User_ID', 'Product_List']

# 将结果存储在dataset中（这里的dataset其实就是处理后的新数据框）
dataset = grouped_data

dataset.head()

In [ ]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori

te = TransactionEncoder()
te_ary = te.fit(dataset).transform(dataset)
te_ary

In [ ]:
df_dd = pd.DataFrame(te_ary, columns = te.columns_)

num_columns = len(df_dd.columns)
new_column_names = [str(i) for i in range(num_columns)]
df_dd.columns = new_column_names

df_dd

In [ ]:
from mlxtend.frequent_patterns import apriori
freq = apriori(df_dd, min_support = 0.01, use_colnames = True)
freq

由于没有输出结果，因此看一看数据是否出问题

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

data_to_plot = []
for col in df_dd.columns:
    value_counts = df_dd[col].value_counts().reindex([0, 1], fill_value=0)
    data_to_plot.append(value_counts.values.reshape(1, -1))
data_to_plot = np.concatenate(data_to_plot, axis=0)

fig, ax = plt.subplots(figsize=(8, 6))

table = ax.table(cellText=data_to_plot,
                 rowLabels=df_dd.columns,
                 colLabels=[0, 1],
                 cellLoc='center',
                 loc='center')

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.5)

ax.axis('off')

plt.savefig('result.png')

In [ ]:
for col in df_dd.columns:
    print(f"Column: {col}")
    print(df_dd[col].value_counts().reindex([0, 1], fill_value=0))
    print("-" * 30)

可见，整体数据较为独立，商品和用户基本为对点购买，因此没有关联规则也不足为奇